# 📦 Structured Output (Salida Estructurada)

En muchas aplicaciones reales, no basta con recibir texto libre de un modelo de lenguaje — necesitamos **datos estructurados** que podamos procesar, almacenar o enviar a otro sistema. Por ejemplo: una receta con ingredientes en una lista, un análisis con campos específicos, o un reporte con formato JSON.

LangChain nos permite definir **esquemas con Pydantic** y usar un `JsonOutputParser` para garantizar que la respuesta del modelo se ajuste exactamente a la estructura que definimos. Esto convierte al LLM en una fuente confiable de datos estructurados.

📄 [Documentación oficial: Structured Output](https://docs.langchain.com/oss/python/langchain/structured-output)

## Configuración

Cargar y/o verificar las variables de entorno necesarias.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

variables_requeridas = ["GEMINI_API_KEY"]
for var in variables_requeridas:
    if os.getenv(var):
        print(f"✅ {var} cargada correctamente")
    else:
        print(f"❌ {var} no encontrada")

## Definir un esquema con Pydantic

El primer paso es crear un modelo de Pydantic que describa la estructura de datos que queremos recibir del LLM. Cada campo tiene un tipo y una descripción — el modelo de lenguaje usa estas descripciones para entender qué debe generar.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Pelicula(BaseModel):
    titulo: str = Field(description="Título de la película")
    director: str = Field(description="Nombre del director")
    año: int = Field(description="Año de estreno")
    generos: List[str] = Field(description="Lista de géneros (ej: acción, comedia, drama)")
    sinopsis: str = Field(description="Sinopsis breve de la película en 2-3 oraciones")

In [ ]:
# Inspeccionar el esquema JSON que genera Pydantic
print(Pelicula.model_json_schema())

## Construir la cadena con `JsonOutputParser`

Ahora conectamos el esquema con un `JsonOutputParser`, creamos un prompt que incluya las instrucciones de formato, y armamos la cadena con la sintaxis pipe: `prompt | llm | parser`.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. Crear el parser con nuestro esquema
json_parser = JsonOutputParser(pydantic_object=Pelicula)

# 2. Crear el prompt e inyectar las instrucciones de formato
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en cine. Responde siempre en español.\n{format_instructions}"),
    ("human", "Dame información sobre la película: {pelicula}")
])

# Inyectar format_instructions automáticamente
prompt = prompt.partial(format_instructions=json_parser.get_format_instructions())

# 3. Crear el modelo
modelo = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)

# 4. Armar la cadena
cadena = prompt | modelo | json_parser

In [ ]:
# Invocar la cadena
resultado = cadena.invoke({"pelicula": "Inception"})
print(resultado)

In [ ]:
# Inspeccionar el tipo del resultado — ya es un diccionario de Python, no texto
print(type(resultado))

In [ ]:
# Acceder a campos individuales como cualquier diccionario
print(f"Título: {resultado['titulo']}")
print(f"Director: {resultado['director']}")
print(f"Año: {resultado['año']}")
print(f"Géneros: {', '.join(resultado['generos'])}")

Observa que el resultado **ya no es texto libre** — es un diccionario de Python con los campos exactos que definimos en nuestro esquema `Pelicula`. Esto nos permite procesarlo programáticamente sin necesidad de parsear texto.

## Esquemas más complejos con validación

Pydantic nos permite agregar restricciones y tipos más específicos. Veamos un ejemplo con `Literal` para valores permitidos y listas con descripciones detalladas.

In [ ]:
from typing import Literal

class Reseña(BaseModel):
    titulo_pelicula: str = Field(description="Título de la película reseñada")
    calificacion: Literal["⭐", "⭐⭐", "⭐⭐⭐", "⭐⭐⭐⭐", "⭐⭐⭐⭐⭐"] = Field(
        description="Calificación de 1 a 5 estrellas"
    )
    puntos_positivos: List[str] = Field(description="Lista de 2-3 aspectos positivos de la película")
    puntos_negativos: List[str] = Field(description="Lista de 1-2 aspectos negativos o mejorables")
    recomendacion: str = Field(description="Una oración indicando a quién le recomendarías esta película")

In [ ]:
# Crear cadena con el nuevo esquema
json_parser_reseña = JsonOutputParser(pydantic_object=Reseña)

prompt_reseña = ChatPromptTemplate.from_messages([
    ("system", "Eres un crítico de cine. Genera reseñas objetivas y balanceadas en español.\n{format_instructions}"),
    ("human", "Genera una reseña de: {pelicula}")
])

prompt_reseña = prompt_reseña.partial(format_instructions=json_parser_reseña.get_format_instructions())

cadena_reseña = prompt_reseña | modelo | json_parser_reseña

In [ ]:
reseña = cadena_reseña.invoke({"pelicula": "Coco de Pixar"})
reseña

In [ ]:
# Acceder a los datos estructurados
print(f"Película: {reseña['titulo_pelicula']}")
print(f"Calificación: {reseña['calificacion']}")
print(f"\nPuntos positivos:")
for punto in reseña["puntos_positivos"]:
    print(f"  ✅ {punto}")
print(f"\nPuntos negativos:")
for punto in reseña["puntos_negativos"]:
    print(f"  ⚠️ {punto}")
print(f"\nRecomendación: {reseña['recomendacion']}")

Nota cómo el campo `calificacion` está restringido a exactamente 5 valores posibles con `Literal`. Pydantic y el parser trabajan juntos para que el modelo genere solo valores válidos.

## Procesamiento en lote (batch)

Una ventaja clave de la salida estructurada es que podemos invocar la cadena múltiples veces y acumular los resultados en una lista para procesarlos juntos.

In [ ]:
import json

peliculas = ["The Matrix", "El Laberinto del Fauno", "Parasite"]

resultados = []
for pelicula in peliculas:
    r = cadena.invoke({"pelicula": pelicula})
    resultados.append(r)
    print(f"- {r['titulo']} ({r['año']})")

print(f"\nTotal de películas procesadas: {len(resultados)}")

In [ ]:
# Los resultados son una lista de diccionarios — fácil de serializar a JSON
print(json.dumps(resultados, indent=2, ensure_ascii=False))

Este patrón de procesamiento en lote es exactamente lo que se usa en `parcial_1/run_batch.py` — invocar la cadena con distintas entradas y guardar los resultados como archivos JSON.

## Actividad

Crea tu propio esquema Pydantic para un dominio que te interese (ej: recetas de cocina, reseñas de videojuegos, perfiles de superhéroes) y construye una cadena que genere datos estructurados.

1. Define tu modelo con al menos 4 campos (incluyendo una `List` y un `Literal`)
2. Crea el `JsonOutputParser` y el prompt
3. Invoca la cadena y muestra los resultados

In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field

# 1. Esquema para reseña de videojuego
class ReseñaVideojuego(BaseModel):
    titulo: str = Field(description="Título del videojuego")
    genero: Literal["Acción", "RPG", "Aventura", "Deportes", "Estrategia", "Simulación", "Terror"] = Field(
        description="Género principal del videojuego"
    )
    plataformas: List[str] = Field(description="Lista de plataformas disponibles (ej: PC, PS5, Switch)")
    calificacion: float = Field(description="Calificación del 0.0 al 10.0")
    puntos_fuertes: List[str] = Field(description="Lista de 2-3 aspectos positivos del juego")
    descripcion: str = Field(description="Descripción breve del videojuego en 2-3 oraciones")

# 2. Parser y cadena
parser_juego = JsonOutputParser(pydantic_object=ReseñaVideojuego)

prompt_juego = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en videojuegos. Responde siempre en español.\n{format_instructions}"),
    ("human", "Dame información sobre el videojuego: {videojuego}")
])
prompt_juego = prompt_juego.partial(format_instructions=parser_juego.get_format_instructions())

cadena_juegos = prompt_juego | modelo | parser_juego

# 3. Invocar y mostrar resultados
reseña = cadena_juegos.invoke({"videojuego": "The Legend of Zelda: Breath of the Wild"})
print(f"Título:       {reseña['titulo']}")
print(f"Género:       {reseña['genero']}")
print(f"Plataformas:  {', '.join(reseña['plataformas'])}")
print(f"Calificación: {reseña['calificacion']}/10")
print(f"\nPuntos fuertes:")
for p in reseña["puntos_fuertes"]:
    print(f"  ✅ {p}")
print(f"\nDescripción: {reseña['descripcion']}")